# Module 4 — Q&A RAG

**Requirement:** Build a RAG pipeline using the customer-support dataset
to answer incoming customer questions.

**Components used (as specified in the task):**
- Vector database: local **FAISS** store (chosen to avoid external
  account setup, as allowed by the task).
- Embeddings: **sentence-transformers** (`all-MiniLM-L6-v2`).
- LLM: **Groq**, using `gpt-oss-120b` (or `gpt-oss-20b`).

**Dataset:** `bitext/Bitext-customer-support-llm-chatbot-training-dataset`
— the `instruction` column is embedded for retrieval, and the paired
`response` column is injected into the generation prompt as grounding
context.

**Note:** You need a free Groq API key from https://console.groq.com to
run the generation step. Set it as an environment variable
`GROQ_API_KEY` before running the last cells.


In [ ]:
# 1. Import libraries
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()  # reads the GROQ_API_KEY from a local .env file 

True

## 2. Load the knowledge base (instruction/response pairs)

In [25]:
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = dataset["train"].to_pandas()

instructions = df["instruction"].tolist()
responses = df["response"].tolist()

print(len(instructions), "knowledge base entries")

26872 knowledge base entries


## 3. Embed the instructions with sentence-transformers

In [26]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

instruction_embeddings = embedding_model.encode(
    instructions, show_progress_bar=True, convert_to_numpy=True
)
instruction_embeddings = instruction_embeddings.astype("float32")

Batches: 100%|██████████| 840/840 [09:53<00:00,  1.41it/s]


## 4. Build the FAISS vector index

In [27]:
dimension = instruction_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(instruction_embeddings)

print("Vectors stored in index:", index.ntotal)

Vectors stored in index: 26872


## 5. Retrieval function (top-k similar chunks)

In [28]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_embedding, k)
    return [responses[i] for i in indices[0]]

## 6. Generation step using Groq with the prompt template

In [29]:
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

PROMPT_TEMPLATE = """System: "You are a helpful, professional customer support assistant
for an online retailer. Answer the customer's question using ONLY
the information in the retrieved support responses below. If the
customer sounds frustrated ({detected_sentiment}), acknowledge
that before answering. If the retrieved context does not cover
the question, say so honestly and offer to escalate to a human
agent rather than guessing."

Context (retrieved past support responses):
{retrieved_chunk_1}
{retrieved_chunk_2}
{retrieved_chunk_3}

Customer question: "{user_message}"
"""

def generate_answer(user_message, detected_sentiment="neutral"):
    retrieved_chunks = retrieve(user_message, k=3)

    prompt = PROMPT_TEMPLATE.format(
        detected_sentiment=detected_sentiment,
        retrieved_chunk_1=retrieved_chunks[0],
        retrieved_chunk_2=retrieved_chunks[1],
        retrieved_chunk_3=retrieved_chunks[2],
        user_message=user_message
    )

    completion = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )

    return completion.choices[0].message.content

## 7. Test the RAG pipeline

In [30]:
answer = generate_answer(
    user_message="I want to cancel my order, this is so frustrating",
    detected_sentiment="frustrated"
)
print(answer)

I’m sorry you’re having to deal with this—let’s get your order cancelled as quickly as possible.

**Here’s how to cancel an order:**

1. **Log in** to your account on our {{Online Company Portal Info}}.  
2. Go to the **‘{{Online Order Interaction}}’** (or the similarly‑named order‑management) section.  
3. Find the order you want to cancel. If you have the order number handy, locate the order with **{{Order Number}}**; otherwise, look through your recent purchases.  
4. Click the option labeled **‘{{Online Order Interaction}}’** to start the cancellation.  
5. Follow any additional prompts (e.g., confirming the reason for cancellation) and submit the request.

If you run into any issues or need further help, you can reach our support team during {{Customer Support Hours}} by phone at {{Customer Support Phone Number}} or via Live Chat on {{Website URL}}.  

Let me know the order number if you have it, and I’ll be glad to confirm the cancellation for you.


## 8. Save the FAISS index and responses (for deployment)

Saving the built index avoids recomputing embeddings for all 26,872
knowledge-base entries every time the deployment server starts.


In [31]:
import pickle

faiss.write_index(index, "faiss_index.index")

with open("rag_responses.pkl", "wb") as f:
    pickle.dump(responses, f)